In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("arkhoshghalb/twitter-sentiment-analysis-hatred-speech")

print("Path to dataset files:", path)

100%|██████████| 1.89M/1.89M [00:01<00:00, 1.42MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/arkhoshghalb/twitter-sentiment-analysis-hatred-speech/versions/1


In [10]:
import os

path = "/root/.cache/kagglehub/datasets/arkhoshghalb/twitter-sentiment-analysis-hatred-speech/versions/1"

print(os.listdir(path))

['test.csv', 'train.csv']


In [11]:
df = pd.read_csv(path + "/train.csv")
print(df.head())

   id  label                                              tweet
0   1      0   @user when a father is dysfunctional and is s...
1   2      0  @user @user thanks for #lyft credit i can't us...
2   3      0                                bihday your majesty
3   4      0  #model   i love u take with u all the time in ...
4   5      0             factsguide: society now    #motivation


In [12]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#\w+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    return text

df['cleaned_text'] = df['tweet'].apply(clean_text)

In [13]:
print(df[['tweet', 'cleaned_text']].head())

                                               tweet  \
0   @user when a father is dysfunctional and is s...   
1  @user @user thanks for #lyft credit i can't us...   
2                                bihday your majesty   
3  #model   i love u take with u all the time in ...   
4             factsguide: society now    #motivation   

                                        cleaned_text  
0    when a father is dysfunctional and is so sel...  
1    thanks for  credit i cant use cause they don...  
2                                bihday your majesty  
3        i love u take with u all the time in ur     
4                         factsguide society now      


In [14]:
df.rename(columns={'class': 'label'}, inplace=True)

In [16]:
from transformers import BertTokenizer, BertForSequenceClassification

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=3
)

model.eval()

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [17]:
text = "AI is changing the world rapidly."
cleaned = clean_text(text)

inputs = tokenizer(
    cleaned,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=128
)

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
predicted_class = torch.argmax(logits, dim=1).item()

print("Input:", text)
print("Cleaned:", cleaned)
print("Prediction:", predicted_class)

Input: AI is changing the world rapidly.
Cleaned: ai is changing the world rapidly
Prediction: 1


In [18]:
labels = {
    0: "Hate Speech",
    1: "Offensive Language",
    2: "Normal"
}

print("Prediction Label:", labels[predicted_class])

Prediction Label: Offensive Language


**What is Transfer Learning and why it is useful?**

Transfer Learning is a technique where a model trained on a large dataset is reused for a different but related task.

**Example:**

BERT is trained on massive text data (Wikipedia, Books)

We reuse it for sentiment analysis / hate speech detection

**Why is it Useful?**

1. Saves Time : No need to train from scratch

2. Better Accuracy: Pretrained models already understand language patterns

3. Works with Small Data: Even limited datasets give good results

4. Less Computation: Training BERT from scratch is very expensive